# 3. Deep stream

Run on Colab with a GPU.

**Needs** `lifs_ged4_hires_train_v2` from notebook 1.

**Makes** (in `MyDrive/CARE2026_ckpts`)
- `lifs_ged4_v2_model/ged4_v2_seed{42,123,7}.pth`: the deep stream trained on scanners A and B1 (B2 held out), three seeds
- `lifs_loco_deep/loco_deep_preds.csv`: leave one scanner out predictions for every examination, three seeds per held out scanner

The model is a ConvNeXt Tiny slice encoder (ImageNet weights from `timm`) with attention pooling over the GED4 slices. In the code, `FairCVaR(0.8, ...)` is the average top k loss of Sec. 3.2 of the paper.

**Step 1.** Mount Google Drive and set the project folder.

In [ ]:
# Settings: mount Google Drive and set the project folder
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"

**Step 2.** Train on A and B1 with seeds 42, 123 and 7 (B2 held out).

In [ ]:
# === Cell D: retrain 3 seeds on uniform ST2 crops (v2) ===
import numpy as np, glob, os, time, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from collections import Counter

HIRES=f"{CKPT}/lifs_ged4_hires_train_v2"   # the new consistent crops
OUT=f"{CKPT}/lifs_ged4_v2_model"; os.makedirs(OUT,exist_ok=True)
all_files=sorted(glob.glob(f"{HIRES}/*.npz"))
vof=lambda f:str(np.load(f)["vendor"])
train_files=[f for f in all_files if vof(f)!="B2"]
print(f"train (A+B1): {len(train_files)} | total: {len(all_files)}")

class DS(Dataset):
    def __init__(s,files,tr=True): s.files=files; s.tr=tr
    def __len__(s): return len(s.files)
    def __getitem__(s,i):
        d=np.load(s.files[i]); x=d["ged4"].astype(np.float32); y=int(d["stage"])-1
        if s.tr:
            if random.random()<.5: x=x[:,:,::-1].copy()
            if random.random()<.5: x=x[:,::-1,:].copy()
        return torch.from_numpy(x.copy()), y

class SliceAttnNet(nn.Module):
    def __init__(s,n=4,b="convnext_tiny"):
        super().__init__(); s.enc=timm.create_model(b,pretrained=True,in_chans=1,num_classes=0,global_pool="avg")
        f=s.enc.num_features; s.attn=nn.Sequential(nn.Linear(f,128),nn.Tanh(),nn.Linear(128,1))
        s.head=nn.Sequential(nn.Dropout(0.5),nn.Linear(f,n))
    def forward(s,x):
        B,S,H,W=x.shape; x=x.reshape(B*S,1,H,W); fe=s.enc(x).reshape(B,S,-1)
        a=torch.softmax(s.attn(fe).squeeze(-1),1); return s.head((fe*a.unsqueeze(-1)).sum(1))

class FairCVaR(nn.Module):
    def __init__(s,alpha=0.8,cw=None): super().__init__(); s.a=alpha; s.cw=cw
    def forward(s,logits,y):
        ce=F.cross_entropy(logits,y,weight=s.cw,reduction="none")
        k=max(1,int(s.a*ce.numel())); w,_=torch.topk(ce,k); return w.mean()

tr_y=[int(np.load(f)["stage"])-1 for f in train_files]
cnt=Counter(tr_y); tot=sum(cnt.values())
cw=torch.tensor([tot/(4*cnt[c]) for c in range(4)],dtype=torch.float32).cuda()

def train(seed,epochs=40):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr=DataLoader(DS(train_files,True),batch_size=4,shuffle=True,num_workers=2,drop_last=True)
    m=SliceAttnNet().cuda(); opt=torch.optim.AdamW(m.parameters(),lr=1e-4,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs)
    lossfn=FairCVaR(0.8,cw); scaler=torch.amp.GradScaler('cuda'); m.train()
    for ep in range(epochs):
        t0=time.time(); tl=0;n=0
        for x,y in tr:
            x,y=x.cuda(),y.cuda(); opt.zero_grad()
            with torch.amp.autocast('cuda'): loss=lossfn(m(x),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl+=loss.item()*x.size(0); n+=x.size(0)
        sched.step()
        if ep%10==0 or ep==epochs-1: print(f"  seed{seed} ep{ep:02d} loss {tl/n:.4f} ({time.time()-t0:.0f}s)")
    p=f"{OUT}/ged4_v2_seed{seed}.pth"; torch.save(m.state_dict(),p); print(f"  ✓ {p}")

for s in [42,123,7]:
    print(f"\n=== seed {s} ===")
    train(s,40)
print("\n✓ all 3 v2 seeds done")

**Step 3.** Train with A held out and with B1 held out. The B2 fold reuses the models above. Saves the leave one scanner out predictions.

In [ ]:
# === LOCO deep: hold out A, then B1; B2 fold reuses the existing 387-models ===
!pip install -q timm 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, time, random
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import roc_auc_score
torch.backends.cudnn.benchmark = True

CKPT  = "/content/drive/MyDrive/CARE2026_ckpts"
HIRES = f"{CKPT}/lifs_ged4_hires_train_v2"
OUT   = f"{CKPT}/lifs_loco_deep"; os.makedirs(OUT, exist_ok=True)
SEEDS = [42, 123, 7]

files = sorted(glob.glob(f"{HIRES}/*.npz"))
VEND, STAGE = {}, {}
for f in files:
    d = np.load(f); VEND[f] = str(d["vendor"]); STAGE[f] = int(d["stage"])
print("vendors:", dict(Counter(VEND.values())))

class DS(Dataset):
    def __init__(s, fl): s.fl = fl
    def __len__(s): return len(s.fl)
    def __getitem__(s, i):
        x = np.load(s.fl[i])["ged4"].astype(np.float32); y = STAGE[s.fl[i]] - 1
        if random.random() < .5: x = x[:, :, ::-1]
        if random.random() < .5: x = x[:, ::-1, :]
        return torch.from_numpy(x.copy()), y

class SliceAttnNet(nn.Module):
    def __init__(s, n=4, pretrained=True):
        super().__init__()
        s.enc = timm.create_model("convnext_tiny", pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")
        f = s.enc.num_features
        s.attn = nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
        s.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(f, n))
    def forward(s, x):
        B, S, H, W = x.shape
        fe = s.enc(x.reshape(B*S, 1, H, W)).reshape(B, S, -1)
        a = torch.softmax(s.attn(fe).squeeze(-1), 1)
        return s.head((fe*a.unsqueeze(-1)).sum(1))

def train(tr_files, seed, epochs=40):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cnt = Counter(STAGE[f]-1 for f in tr_files); tot = sum(cnt.values())
    cw = torch.tensor([tot/(4*max(cnt[c], 1)) for c in range(4)], dtype=torch.float32).cuda()
    dl = DataLoader(DS(tr_files), batch_size=4, shuffle=True, num_workers=2, drop_last=True)
    m = SliceAttnNet().cuda()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs); sc = torch.amp.GradScaler('cuda'); m.train()
    for ep in range(epochs):
        t0 = time.time()
        for x, y in dl:
            x, y = x.cuda(), y.cuda(); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                ce = F.cross_entropy(m(x), y, weight=cw, reduction="none")
                loss = torch.topk(ce, max(1, int(0.8*ce.numel()))).values.mean()   # FairCVaR, as deployed
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step()
        if ep % 10 == 0 or ep == epochs-1: print(f"    seed{seed} ep{ep:02d} ({time.time()-t0:.0f}s)")
    return m.eval()

@torch.no_grad()
def predict(m, fl):
    out = []
    for f in fl:
        x = np.load(f)["ged4"].astype(np.float32)
        P = [F.softmax(m(torch.from_numpy(v.copy())[None].cuda().float()), 1)[0].cpu().numpy()
             for v in (x, x[:, :, ::-1], x[:, ::-1, :], x[:, ::-1, ::-1])]
        out.append(np.mean(P, 0))
    return np.array(out)

rows = []
for held in ['A', 'B1', 'B2']:
    te = [f for f in files if VEND[f] == held]
    tr = [f for f in files if VEND[f] != held]
    print(f"\n=== held-out {held}: train {len(tr)}, test {len(te)} ===")
    for s in SEEDS:
        pp = f"{OUT}/preds_{held}_seed{s}.npy"
        if os.path.exists(pp):
            P = np.load(pp); print(f"  seed {s}: cached")
        else:
            if held == 'B2':
                m = SliceAttnNet(pretrained=False).cuda()
                m.load_state_dict(torch.load(f"{CKPT}/lifs_ged4_v2_model/ged4_v2_seed{s}.pth")); m.eval()
            else:
                mp = f"{OUT}/model_{held}_seed{s}.pth"
                if os.path.exists(mp):
                    m = SliceAttnNet(pretrained=False).cuda(); m.load_state_dict(torch.load(mp)); m.eval()
                else:
                    m = train(tr, s); torch.save(m.state_dict(), mp)
            P = predict(m, te); np.save(pp, P); del m; torch.cuda.empty_cache()
        st = np.array([STAGE[f] for f in te])
        print(f"  seed {s}: sub {roc_auc_score(st>=2, 1-P[:,0]):.3f}  cir {roc_auc_score(st>=4, P[:,3]):.3f}")
        for f, p in zip(te, P):
            rows.append(dict(filename=os.path.basename(f)[:-4], held=held, seed=s, stage=STAGE[f],
                             d1=p[0], d2=p[1], d3=p[2], d4=p[3]))

pd.DataFrame(rows).to_csv(f"{OUT}/loco_deep_preds.csv", index=False)
print(f"\nsaved {len(rows)} rows -> {OUT}/loco_deep_preds.csv")